In [1]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content = "hello world",
    metadata = {"source": "https://www.flipkart.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'https://www.flipkart.com'}, page_content='hello world')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [6]:
from langchain_community.document_loaders.pdf import PyPDFLoader

pdf_loader = PyPDFLoader("data/data cleaning.pdf")
document = pdf_loader.load()




C:\Users\karth\AppData\Local\Temp\ipykernel_19564\1465853068.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader



PyPDFLoader ===> is used for the simple pdf's
PymuPDFLoader ====> is used for the complex pdf like images ,graphs etc

 # ingestion pipline 

In [7]:
# data ==> documents

import os
from langchain_community.document_loaders.pdf import PyPDFLoader


In [8]:
def load_pdf():
    folder_path = "data/"
    num_docs = 0
    all_docs = []
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path,filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs: ", num_docs)
    print("total pages: ", len(all_docs))
    return all_docs
    


In [9]:
all_pdf_documents = load_pdf()

total pdfs:  2
total pages:  32


## now all the data is converted into documents

## chunking

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)

    return chunked_docs

In [11]:
print(split_docs)

<function split_docs at 0x00000136830FFA60>


In [12]:
import inspect
print(inspect.signature(split_docs))

(documents, chunk_size=500, chunk_overlap=50)


In [13]:
print(inspect.signature(split_docs))

(documents, chunk_size=500, chunk_overlap=50)


In [14]:
print(type(all_pdf_documents))

<class 'list'>


In [15]:
# data ==> documents

import os
from langchain_community.document_loaders.pdf import PyPDFLoader

def load_pdf():
    folder_path = "data/"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))

    return all_docs


# Load PDFs
all_pdf_documents = load_pdf()


# Split documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)

    return chunked_docs


# IMPORTANT: Don't use ()
chunks = split_docs(all_pdf_documents)

print("Total chunks:", len(chunks))

total pdfs: 2
total pages: 32
Total chunks: 56


## embeddings

In [16]:
# converts the chunks into embeddings 

In [17]:
from sentence_transformers import SentenceTransformer

In [18]:
class embedding_manager:
    def __init__(self,model_name = "all-MiniLM-L6-v2"):
        self.Model_name = model_name
        print("loading model....", self.Model_name)
        self.model = SentenceTransformer(self.Model_name)
        print("embedding dimensions= ",self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self,text):
        embeddings = self.model.encode(text,show_progress_bar = True)
        print("embeddings shape: ", embeddings.shape)
        return embeddings
        
embedding_manager = embedding_manager()
print("completed")

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions=  384
completed


C:\Users\karth\AppData\Local\Temp\ipykernel_19564\3125792225.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions= ",self.model.get_sentence_embedding_dimension())


In [19]:
import chromadb
import uuid

In [20]:
class Vectorstoremanager:
    def __init__(self, persist_directory ="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initalize_store()
    def _initalize_store(self):
        os.makedirs(self.persist_directory, exist_ok = True)

        self.client = chromadb.PersistentClient(path = self.persist_directory)

        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata = {"description": "vector store collection for pdf embeddings in RAG"}
        )
        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection: ", self.collection.count())

        
#     def add_documents(self, documents, embeddings):
#         if len(documents) != len(embeddings):
#             raise ValueError("number of documents does not match number of embeddings")

#         # storing => ids, embeddings, documents, metadata
#         ids = []
#         all_metadata = []
#         documents_content = []
#         embeddings = []

#         for i ,(doc,embedding) in enumerate(zip(documents,embeddings)):
#             doc_id = f"doc_{uuid.uuid()}"
#             ids.append(doc_id)

#             metadata = dict(doc.metadata)
#             metadata["doc_index"] = i
#             metadata["content_length"] = len(doc.page_content)
#             all_metadata.append(metadata)

#             documents_content.append(doc.apge_content)
#             embeddings_list.append(embeddings.tolist())
#             self.collection.add(
#                 ids =ids,
#                 metadatas = all_metadata,
#                 documents = documents_content,
#                 embeddings = embeddings_list
#             )
#         print("total documents added in vector store= ", len(documents_content))
#         print("docs in collections: ", self.collection.count())


    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("number of documents does not match number of embeddings")
    
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []
    
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)
    
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)
    
            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
    
        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )
    
        print("total documents added in vector store=", len(documents_content))
        print("docs in collections:", self.collection.count())

In [21]:
vector_store.client.delete_collection(name="pdf_documents")
vector_store = Vectorstoremanager()  # recreates it fresh

NameError: name 'vector_store' is not defined

In [23]:
texts = [doc.page_content for doc in chunks]

embeddings_list = embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks,embeddings_list)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

embeddings shape:  (56, 384)


NameError: name 'vector_store' is not defined

In [27]:
from sentence_transformers import SentenceTransformer
import chromadb
import hashlib
import os


# ============================
# Embedding Manager
# ============================
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, texts):
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print("embeddings shape: ", embeddings.shape)
        return embeddings


# ============================
# Vector Store Manager
# ============================
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        self.client = chromadb.PersistentClient(path=self.persist_directory)

        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )
        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def _make_id(self, doc, index):
        """Deterministic ID based on content, so re-running ingestion never creates duplicates."""
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "unknown")
        raw = f"{source}_{page}_{index}_{doc.page_content}"
        return hashlib.md5(raw.encode("utf-8")).hexdigest()

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("number of documents does not match number of embeddings")

        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = self._make_id(doc, i)
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        # upsert instead of add: same content -> same id -> overwritten, not duplicated
        self.collection.upsert(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collections:", self.collection.count())

    def reset(self):
        """Wipe the collection completely and recreate it empty."""
        self.client.delete_collection(name=self.collection_name)
        self._initialize_store()


# ============================
# Run it
# ============================
embedding_manager = EmbeddingManager()
vector_store = VectorStoreManager()

texts = [doc.page_content for doc in chunks]
embeddings_list = embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks, embeddings_list)

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\karth\AppData\Local\Temp\ipykernel_19564\2431506592.py:15: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


embedding dimensions= 384
initialized the vector store with collection: pdf_documents
docs in collection: 112


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

embeddings shape:  (56, 384)
total documents added in vector store= 56
docs in collections: 112


## retriever pipeline

In [28]:
from sklearn.metrics.pairwise import cosine_similarity

In [43]:
class RAGRetriever:
    def __init__(self,embedding_manager,vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self,query, top_k = 5, score_threshold = 0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        # cosine similarity
        retrived_docs = []
        if results["documents"] and results["documents"] [0]:
            ids = results["ids"] [0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i ,(doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrived_docs.append({
                        "id": doc_id,
                        "document": document,
                        "distance" : distance,
                        "similarity_score": similarity_score,
                        "rank" : i+1
                    })

                print(f"retrived{len(retrived_docs)} documents")
            else:
                print("no documents found")
            return retrived_docs
print("completeed")

completeed


In [44]:
rag_retriver= RAGRetriever(embedding_manager, vector_store)

In [46]:
rag_retriver.retrieve("what is the ifnull() ?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape:  (1, 384)
retrived1 documents
retrived2 documents
retrived3 documents
retrived4 documents
retrived5 documents
no documents found


[{'id': 'doc_cfdffe54-d48c-4581-ab98-c0dfd309ca77',
  'document': "1.3 - ISNULL():  \nThe ISNULL() function in SQL is used to check if a value is NULL. It returns a specified value if the \noriginal value is NULL. It's similar to IFNULL() but is more common in SQL Server.\nIFNULL() and ISNULL() are simpler and more focused, but support only two arguments.\nExample:\nSuppose you have a table called employees with columns for base_salary, bonus, and \ntotal_compensation, where some values in bonus and total_compensation might be NULL.",
  'distance': 0.8298285007476807,
  'similarity_score': 0.17017149925231934,
  'rank': 1},
 {'id': 'b41d19cdff990ec0816de82bf49bb760',
  'document': "1.3 - ISNULL():  \nThe ISNULL() function in SQL is used to check if a value is NULL. It returns a specified value if the \noriginal value is NULL. It's similar to IFNULL() but is more common in SQL Server.\nIFNULL() and ISNULL() are simpler and more focused, but support only two arguments.\nExample:\nSuppose

# intergrate with LLM's

## groq 

In [71]:
api_key_groq = "API KEY"

In [72]:
# !pip install langchain
# !pip install -U langchain-groq

In [83]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key = api_key_groq,
    model = "openai/gpt-oss-120b",
    temperature = 0.7,
    max_tokens = 512
)

In [84]:
# generating the retriveral augemented outputs

# def generate_output(query, retriever, llm, top_k = 3):
#     results = retriever.retrieve(query, top_k)
#     context = "\n".join([doc["document"] for doc in results]) if results else ""
    
#     if not context:
#         print("we found no relevant for the given query")
        
#         # context+ query
#         prompt = f""" use given context to generate the answer for the query
#                     Context: {context}
#                     Query: {query} """
#         response = llm.invoke([prompt.format(context= context, query = query)])
#         return response.content

def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)
    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant docs for the given query")

    prompt = f"""Use the given context to answer the query.
Context: {context}
Query: {query}"""

    response = llm.invoke(prompt)
    return response.content

In [90]:
answer = generate_output(" what is data cleaning ?" , rag_retriver,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape:  (1, 384)
retrived1 documents
retrived2 documents
retrived3 documents
no documents found


In [91]:
print(answer)

**Data cleaning** is the process of detecting and fixing errors, inconsistencies, and inaccuracies in a dataset so that the data becomes reliable and ready for analysis.  

When you perform data cleaning in **SQL**, you typically:

- Identify problems such as duplicate rows, missing values, incorrect data types, out‑of‑range values, and inconsistent formatting.  
- Apply SQL operations (e.g., `SELECT … WHERE`, `UPDATE`, `DELETE`, `JOIN`, `CASE`, string functions, date functions, etc.) to correct or remove the faulty data.  
- Standardize values (e.g., unify date formats, trim whitespace, enforce consistent case).  
- Fill or flag missing data, and resolve duplicates.

The goal is to transform raw, messy data into a **clean, consistent, and accurate** dataset that can be trusted for reporting, analytics, or any downstream decision‑making. In short, data cleaning turns raw data into actionable insights using SQL.
